# Example: HMM-Powered Backtesting

In this example, we fit a JumpHMM model to synthetic training data, generate regime-switching test paths, and backtest the Session 2 **Cobb-Douglas rebalancing engine** across hundreds of Monte Carlo scenarios. We visualize the distribution of outcomes and identify where the engine succeeds and fails.

> __Learning Objectives:__
>
> * __Fit and tune a JumpHMM model from price data:__ Use the JumpHMM.jl package to fit a Hidden Markov Model to synthetic training prices and tune its jump parameters. Verify that the fitted model produces regime-switching dynamics with realistic volatility clustering.
> * __Generate regime-switching synthetic market paths:__ Use the fitted HMM to create hundreds of Monte Carlo price paths that exhibit diverse market conditions. Combine the HMM market paths with SIM-based per-ticker price generation to build a full backtest scenario.
> * __Backtest a rebalancing engine and analyze outcome distributions:__ Run the Cobb-Douglas rebalancing engine and a buy-and-hold benchmark across all Monte Carlo paths. Compare the distributions of final wealth, max drawdown, and Sharpe ratio to assess strategy robustness.

Let's dive in!
___

## Setup, Data and Prerequisites
We begin by loading the `eCornellAIFinance` package and session dependencies via `Include.jl`.

In [ ]:
# Load the eCornellAIFinance package and session-3 dependencies
include("Include.jl");

In [ ]:
let
    # --- Step 1: Define time and model constants ---
    global Δt = 1.0 / 252.0;       # daily time step in years (1 trading day = 1/252 year)
    global rf = 0.05;               # annualized risk-free rate
    global N_states = 50;           # number of hidden states in the HMM
    global ν = 5.0;                 # Student-t degrees of freedom for emission model
    global B₀ = 10000.0;           # initial portfolio budget ($)
    global N_short = 21;            # short moving average window (approx. 1 month)
    global N_long = 63;             # long moving average window (approx. 3 months)
    global offset = N_short + N_long; # warmup period for moving average indicators
    global n_mc_paths = 100;        # number of Monte Carlo paths (increase for production)
    global n_trading_days = 336;    # offset + 252 trading days (1 year of active trading)

    # --- Step 2: Define asset universe and SIM parameters (same as Session 2) ---
    # Each ticker has (alpha, beta, sigma_epsilon) for the Single Index Model
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    # --- Step 3: Define starting prices for each asset ---
    global start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    println("Setup complete: $(n_mc_paths) MC paths, $(N_states) HMM states, ν = $(ν)")
end

___
## Task 1: Fit and Tune the JumpHMM Model
We generate 5 years of synthetic "training" prices (using GBM as a stand-in for historical data), then fit a JumpHMM model and tune its jump parameters to match the training data's kurtosis and autocorrelation structure.

> __What should you see?__
>
> The fitted model will have a transition matrix that shows regime persistence (strong diagonal), emission distributions that vary across states (low-volatility calm states vs. high-volatility crisis states), and tuned jump parameters (ε, λ) that produce realistic volatility clustering.

In the next cell, we call [the `generate_training_prices(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.generate_training_prices) to create synthetic GBM prices, then fit and tune the HMM using `hmm_fit` and `hmm_tune` from [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl). The fitted model is stored in the `hmm_model` variable.

In [ ]:
let
    # --- Step 1: Generate 5 years of synthetic GBM training data ---
    # This serves as a stand-in for historical price data used to fit the HMM.
    # Parameters: S₀ = initial price, μ = drift, σ = volatility, T = number of days.
    training_prices = generate_training_prices(S₀=100.0, μ=0.08, σ=0.18, T=1260, seed=42);

    # --- Step 2: Fit the JumpHMM model to the training prices ---
    # The model learns N_states hidden states with Student-t emission distributions.
    # Each state has its own mean and variance, capturing different market regimes.
    println("Fitting JumpHMM model (N = $(N_states) states)...")
    global hmm_model = hmm_fit(JumpHiddenMarkovModel, training_prices; 
        rf=rf, N=N_states, nu=ν, dt=Δt);
    
    # --- Step 3: Tune jump parameters (ε, λ) via grid search ---
    # ε = jump probability per step, λ = mean jump duration.
    # The tuner searches over a grid and selects the (ε, λ) pair that best matches
    # the empirical kurtosis and autocorrelation structure of the training data.
    println("Tuning jump parameters (ε, λ)...")
    global hmm_model = hmm_tune(hmm_model, training_prices;
        epsilon_range=range(1e-4, 2.5e-2, length=10),
        lambda_range=range(10.0, 100.0, length=8),
        n_paths=50);
    
    # --- Step 4: Report fitted model parameters ---
    println("\nModel fitted and tuned:")
    println("  Jump probability (ε): $(round(hmm_model.jump.epsilon, digits=5))")
    println("  Jump duration mean (λ): $(round(hmm_model.jump.lambda, digits=1))")
    println("  Number of states: $(N_states)")
    println("  Training data: $(length(training_prices)) days (5 years)")

    # --- Step 5: Save the fitted model to disk for reuse ---
    save(joinpath(_PATH_TO_DATA, "hmm-model.jld2"), Dict("model" => hmm_model));
end

**Visualize:** A few sample HMM-generated price paths vs. the GBM training data.

> __What should you see?__
>
> The HMM paths should look qualitatively different from GBM. They will exhibit periods of low volatility punctuated by sudden, persistent drops or rallies (the jump mechanism). Some paths will show severe drawdowns that GBM would never produce.

In the next cell, we use `hmm_simulate` from [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) to generate sample paths and plot them alongside a GBM reference path.

In [ ]:
let
    # --- Step 1: Setup visualization parameters ---
    n_sample = 5;   # number of HMM sample paths to generate
    P₀ = 100.0;    # initial price for all paths

    # --- Step 2: Create the base plot ---
    p = plot(size=(800, 450), title="Sample HMM-Generated Market Paths",
        xlabel="Trading Day", ylabel="Price (\$)", legend=:topright, alpha=0.8,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)

    # --- Step 3: Generate and plot HMM sample paths ---
    # Each path is simulated independently from the fitted model.
    # The hmm_simulate function returns growth rates; we convert to prices.
    for i in 1:n_sample
        result = hmm_simulate(hmm_model, n_trading_days; n_paths=1);
        G = result.paths[1].observations;  # extract growth rate observations
        prices = JumpHMM.prices_from_growth_rates(G, P₀; rf=rf, dt=Δt);  # convert to price levels
        plot!(p, 1:length(prices), prices, label=(i == 1 ? "HMM paths" : ""), 
            linewidth=1.2, color=:steelblue, alpha=0.5)
    end

    # --- Step 4: Add a GBM reference path for comparison ---
    # GBM produces smooth, log-normal paths without regime switches or jumps.
    gbm_prices = generate_training_prices(S₀=P₀, μ=0.08, σ=0.18, T=n_trading_days, seed=99);
    plot!(p, 1:length(gbm_prices), gbm_prices, label="GBM reference", 
        linewidth=2, color=:coral, linestyle=:dash)
    
    # --- Step 5: Mark the warmup boundary ---
    # Trading signals are not generated during the warmup period (first `offset` days).
    vline!(p, [offset], label="Warmup end", linestyle=:dot, color=:black, alpha=0.5)
    p
end

___
## Task 2: Generate the Backtest Scenario
We use the fitted HMM to generate a full backtest scenario: $n\_mc\_paths$ market paths, each with per-ticker prices generated via SIM. This creates the Monte Carlo test bed for strategy evaluation.

> __What should you see?__
>
> A fan of price paths showing the range of possible outcomes. The paths should exhibit diverse behaviors (some bullish, some crashing, some sideways), reflecting the full regime-switching dynamics of the HMM.

In the next cell, we call [the `generate_hmm_scenario(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.generate_hmm_scenario) to produce a `MyBacktestScenario` containing all Monte Carlo paths. The result is stored in the `scenario` variable.

In [ ]:
let
    # --- Step 1: Generate the full backtest scenario ---
    # The generate_hmm_scenario function:
    #   1. Uses the fitted HMM to simulate n_mc_paths market index paths
    #   2. For each market path, generates per-ticker prices via the Single Index Model (SIM)
    #   3. Packages everything into a MyBacktestScenario struct
    println("Generating backtest scenario: $(n_mc_paths) paths × $(n_trading_days) days...")

    global scenario = generate_hmm_scenario(hmm_model, tickers, sim_params;
        n_paths=n_mc_paths, n_steps=n_trading_days, P₀_market=100.0,
        start_prices=start_prices, rf=rf, Δt=Δt, label="HMM Regime-Switching");

    # --- Step 2: Verify and report scenario dimensions ---
    println("Scenario generated: $(scenario.n_paths) paths × $(scenario.n_steps) days × $(length(tickers)) tickers")

    # --- Step 3: Save scenario to disk for use in Example 2 (Bandit Challenger) ---
    save(joinpath(_PATH_TO_DATA, "backtest-scenario.jld2"), Dict("scenario" => scenario));
end

___
## Task 3: Backtest the Cobb-Douglas Engine Across All Paths
We run the Session 2 Cobb-Douglas rebalancing engine (with moderate trigger rules: DD ≤ 15%, τ ≤ 50%) and the buy-and-hold benchmark across all Monte Carlo paths. We collect the distribution of final wealth, max drawdown, and Sharpe ratio.

> __What should you see?__
>
> Histograms showing the spread of outcomes. The engine should have a tighter distribution (less variance) than buy-and-hold due to its trigger rules, but some paths will still show losses. The question is how often and how bad.

In the next cell, we call [the `backtest_engine(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.backtest_engine) and [the `backtest_buyhold(...)` function](../../code/docs/build/session3.html#eCornellAIFinance.backtest_buyhold) to run both strategies across all paths. The results are stored in the `engine_bt::MyBacktestResult` and `bh_bt::MyBacktestResult` variables.

In [ ]:
let
    # --- Step 1: Backtest the Cobb-Douglas rebalancing engine ---
    # The engine uses moving average crossover signals with drawdown and turnover triggers.
    # rules_params sets the maximum drawdown (15%) and maximum turnover (50%) thresholds.
    println("Backtesting Cobb-Douglas Engine across $(n_mc_paths) paths...")
    rules_params = (max_drawdown = 0.15, max_turnover = 0.50);
    global engine_bt = backtest_engine(scenario, tickers, sim_params, rules_params;
        B₀=B₀, offset=offset);

    # --- Step 2: Backtest the buy-and-hold benchmark ---
    # Equal-weight buy-and-hold: allocate B₀ equally across all tickers at the start,
    # then hold without rebalancing for the entire trading period.
    println("Backtesting Buy-and-Hold across $(n_mc_paths) paths...")
    global bh_bt = backtest_buyhold(scenario, tickers; B₀=B₀, offset=offset);

    # --- Step 3: Compute and display summary statistics ---
    println("\n" * "═"^55)
    println("  BACKTEST RESULTS: $(n_mc_paths) HMM paths")
    println("═"^55)

    summary = DataFrame(
        "Metric" => ["Median Final Wealth", "Median Max Drawdown (%)", 
            "Median Sharpe", "Failure Rate (% below \$$(Int(B₀)))"],
        "Cobb-Douglas Engine" => [
            round(median(engine_bt.final_wealth), digits=0),
            round(median(engine_bt.max_drawdowns) * 100, digits=1),
            round(median(engine_bt.sharpe_ratios), digits=3),
            round(sum(engine_bt.final_wealth .< B₀) / n_mc_paths * 100, digits=1)
        ],
        "Buy-and-Hold" => [
            round(median(bh_bt.final_wealth), digits=0),
            round(median(bh_bt.max_drawdowns) * 100, digits=1),
            round(median(bh_bt.sharpe_ratios), digits=3),
            round(sum(bh_bt.final_wealth .< B₀) / n_mc_paths * 100, digits=1)
        ]
    );
    pretty_table(summary, tf=tf_markdown)

    # --- Step 4: Save results for use in Example 2 (Bandit Challenger) ---
    save(joinpath(_PATH_TO_DATA, "backtest-results.jld2"),
        Dict("engine" => engine_bt, "buyhold" => bh_bt));
end

**Visualize:** Distribution of final wealth and max drawdown for both strategies.

> __What should you see?__
>
> Two overlapping histograms. The engine's distribution should be shifted right (higher wealth) or have a thinner left tail (fewer catastrophic losses) compared to buy-and-hold. The drawdown distribution should show the engine's trigger rules capping extreme losses.

In the next cell, we plot side-by-side histograms comparing the Cobb-Douglas engine and buy-and-hold outcomes.

In [ ]:
let
    # --- Step 1: Plot the distribution of final wealth (left panel) ---
    # Overlapping histograms show how each strategy's terminal wealth is distributed.
    p1 = histogram(engine_bt.final_wealth, label="Cobb-Douglas Engine", alpha=0.6, 
        color=:steelblue, bins=20,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)
    histogram!(p1, bh_bt.final_wealth, label="Buy-and-Hold", alpha=0.4, color=:coral, bins=20)
    vline!(p1, [B₀], label="Starting Capital", linestyle=:dash, color=:black, linewidth=2)
    xlabel!(p1, "Final Wealth (\$)")
    ylabel!(p1, "Count")
    title!(p1, "Distribution of Final Wealth")

    # --- Step 2: Plot the distribution of max drawdown (right panel) ---
    # Max drawdown measures the largest peak-to-trough decline during each path.
    # The engine's trigger rules should cap extreme drawdowns relative to buy-and-hold.
    p2 = histogram(engine_bt.max_drawdowns .* 100, label="Cobb-Douglas Engine", alpha=0.6, 
        color=:steelblue, bins=20,
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent)
    histogram!(p2, bh_bt.max_drawdowns .* 100, label="Buy-and-Hold", alpha=0.4, color=:coral, bins=20)
    xlabel!(p2, "Max Drawdown (%)")
    ylabel!(p2, "Count")
    title!(p2, "Distribution of Max Drawdown")

    # --- Step 3: Combine into a side-by-side layout ---
    plot(p1, p2, layout=(1, 2), size=(900, 400), legend=:topright)
end

___
## Summary
We fitted a JumpHMM model to synthetic training data, generated hundreds of regime-switching Monte Carlo paths, and backtested the Cobb-Douglas rebalancing engine against a buy-and-hold benchmark.

> __Key Takeaways:__
>
> * **JumpHMM fitting is a two-step process:** First fit the model to training data, then tune jump parameters to match empirical kurtosis and autocorrelation. The resulting model generates paths with realistic regime transitions and volatility clustering.
> * **Distributional backtesting exposes tail risk:** Running the Cobb-Douglas engine across hundreds of HMM paths reveals the spread of terminal wealth, drawdown, and Sharpe outcomes. This includes the worst-case scenarios that matter most for risk management.
> * **The engine handles regime shifts but has limits:** Trigger rules reduce extreme drawdowns compared to buy-and-hold, but performance degrades during sustained crisis regimes. This motivates the bandit challenger we introduce in the next example.

These results establish a quantitative baseline for the Cobb-Douglas engine under regime-switching conditions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.